<a href="https://colab.research.google.com/github/sunchubhanuprakash-coder/electricity-theft-detection-ml/blob/main/mimiproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


STEP 3: Load Dataset

In [ ]:
df = pd.read_csv('/content/data.csv')   # change name if needed
print("Original shape:", df.shape) # NAn is missing  val
df.head()


Original shape: (4640, 1036)


,CONS_NO,FLAG,2014/1/1,2014/1/10,2014/1/11,2014/1/12,2014/1/13,2014/1/14,2014/1/15,2014/1/16,...,2016/9/28,2016/9/29,2016/9/3,2016/9/30,2016/9/4,2016/9/5,2016/9/6,2016/9/7,2016/9/8,2016/9/9
0,0387DD8A07E07FDA6271170F86AD9151,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,10.12,9.96,16.92,7.60,27.22,18.05,26.47,18.75,17.84,14.92
1,01D6177B5D4FFE0CABA9EF17DAFC2B84,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,4B75AC4F2D8434CFF62DB64D0BB43103,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,B32AC8CC6D5D805AC053557AB05F5343,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6.50,9.99,11.78,18.59,26.80,18.57,14.59,12.82,19.37,15.92
4,EDFC78B07BA2908B3395C4EB2304665E,1,2.9,3.42,3.81,4.58,3.56,4.25,3.86,3.53,...,17.77,10.37,15.32,13.51,12.23,14.68,16.35,18.14,18.41,17.31


STEP 4: Convert Values to Numeric & Handle Missing Values


In [ ]:
# Convert all except consumer ID to numeric
df.iloc[:, 1:] = df.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

# Fill missing values using column mean, only considering numeric columns
df.fillna(df.mean(numeric_only=True), inplace=True)

print("After cleaning:", df.shape)


After cleaning: (4640, 1036)


STEP 5: Remove FLAG from Feature Set


In [ ]:
if 'FLAG' in df.columns:
    df.drop(columns=['FLAG'], inplace=True)


STEP 6: Convert Wide Data → Time-Series (Long Format)

In [ ]:
id_col = df.columns[0]          # Consumer ID
date_cols = df.columns[1:]      # Daily readings

df_long = df.melt(
    id_vars=id_col,
    value_vars=date_cols,
    var_name='Date',
    value_name='Consumption'
)

df_long['Date'] = pd.to_datetime(df_long['Date'], errors='coerce')
df_long.dropna(subset=['Date'], inplace=True)

print("After time-series conversion:", df_long.shape)
df_long.head()


After time-series conversion: (4797760, 3)


,CONS_NO,Date,Consumption
0,0387DD8A07E07FDA6271170F86AD9151,2014-01-01,15.377114
1,01D6177B5D4FFE0CABA9EF17DAFC2B84,2014-01-01,15.377114
2,4B75AC4F2D8434CFF62DB64D0BB43103,2014-01-01,15.377114
3,B32AC8CC6D5D805AC053557AB05F5343,2014-01-01,15.377114
4,EDFC78B07BA2908B3395C4EB2304665E,2014-01-01,2.900000


STEP 7: Sort Time-Series Data

In [ ]:
df_long.sort_values(by=[id_col, 'Date'], inplace=True)
df_long.reset_index(drop=True, inplace=True)


STEP 8: Feature Extraction (Previous Consumption)

In [ ]:
df_long['Prev_Consumption'] = (
    df_long.groupby(id_col)['Consumption'].shift(1)
)

df_long['Prev_Consumption'] = df_long['Prev_Consumption'].bfill()


STEP 9: Sudden Drop Percentage Calculation

In [ ]:
# Create a safe denominator where 0s are replaced with NaN to prevent ZeroDivisionError
safe_denominator = df_long['Prev_Consumption'].replace(0, np.nan)

# Perform the division. Division by NaN will result in NaN, preventing ZeroDivisionError.
df_long['Drop_Percentage'] = (df_long['Prev_Consumption'] - df_long['Consumption']) / safe_denominator

# Fill all resulting NaN values (from division by original 0s and any other NaNs) with 0.
df_long['Drop_Percentage'] = df_long['Drop_Percentage'].fillna(0)

# Ensure no inf/-inf values remain (though this method should primarily produce NaNs then 0s)
df_long = df_long.replace([np.inf, -np.inf], 0)

STEP 10: Label Generation (UNLABELED → LABELED)

In [ ]:
def detect_risk(drop):
    if drop < 0.2:
        return 'Normal'
    elif drop < 0.5:
        return 'Potential Theft'
    else:
        return 'Abnormal'

df_long['Risk_Label'] = df_long['Drop_Percentage'].apply(detect_risk)
df_long['Risk_Label'].value_counts()


,count
Risk_Label,
Normal,4191302
Potential Theft,442012
Abnormal,164446


STEP 11: Encode Labels

In [ ]:
le = LabelEncoder()
df_long['Risk_Encoded'] = le.fit_transform(df_long['Risk_Label'])


STEP 12: Feature Selection

In [ ]:
X = df_long[['Consumption', 'Prev_Consumption', 'Drop_Percentage']]
y = df_long['Risk_Encoded']


STEP 13: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


STEP 14: Train Random Forest Model

In [ ]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 0.9999843677049289
                 precision    recall  f1-score   support

       Abnormal       1.00      1.00      1.00     32801
         Normal       1.00      1.00      1.00    838394
Potential Theft       1.00      1.00      1.00     88357

       accuracy                           1.00    959552
      macro avg       1.00      1.00      1.00    959552
   weighted avg       1.00      1.00      1.00    959552



STEP 15: Model Evaluation

In [15]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))


Accuracy: 0.9999843677049289
                 precision    recall  f1-score   support

       Abnormal       1.00      1.00      1.00     32801
         Normal       1.00      1.00      1.00    838394
Potential Theft       1.00      1.00      1.00     88357

       accuracy                           1.00    959552
      macro avg       1.00      1.00      1.00    959552
   weighted avg       1.00      1.00      1.00    959552



STEP 16: User Input Prediction

In [16]:
def predict_theft(consumption, prev_consumption):
    drop = (prev_consumption - consumption) / prev_consumption
    pred = model.predict([[consumption, prev_consumption, drop]])
    return le.inverse_transform(pred)[0]

# Example for test pervious  and current
predict_theft(1299, 2655)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


'Abnormal'

🔹 STEP 17: Transformer-Wise Analysis

In [17]:
# Simulate Transformer IDs
df_long['Transformer_ID'] = np.random.randint(1001, 1050, size=len(df_long))

transformer_risk = (
    df_long[df_long['Risk_Label'] != 'Normal']
    .groupby('Transformer_ID')
    .size()
    .sort_values(ascending=False)
)

transformer_risk.head()


,0
Transformer_ID,
1029,12589
1048,12569
1015,12547
1006,12532
1007,12532


STEP 18: Location-Wise Analysis

In [18]:
# Simulate Locations
locations = ['Hyderabad', 'Warangal', 'Karimnagar', 'Siddipet']
df_long['Location'] = np.random.choice(locations, size=len(df_long))

location_risk = (
    df_long[df_long['Risk_Label'] != 'Normal']
    .groupby('Location')
    .size()
    .sort_values(ascending=False)
)

location_risk


,0
Location,
Siddipet,152090
Hyderabad,151647
Karimnagar,151508
Warangal,151213


Done by @Bhanu Prakash